## Importing Data:

In [9]:
import pandas as pd

train_path = '/kaggle/input/competitions/playground-series-s6e3/train.csv'
test_path = '/kaggle/input/competitions/playground-series-s6e3/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print('Train Shape:', train_df.shape)
display(train_df.head(3))
print('Test Shape:', test_df.shape)
display(test_df.head(3))

Train Shape: (594194, 21)


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.1,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.5,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.4,5841.35,No


Test Shape: (254655, 20)


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,594194,Female,0,Yes,No,72,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Electronic check,115.55,8061.50
1,594195,Female,0,Yes,No,71,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),19.80,1336.50
2,594196,Male,0,No,No,12,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),55.55,633.55


In [10]:
target = 'Churn' #atarget variable

cat_cols = ['gender','Partner', 'Dependents', 'SeniorCitizen',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',] 

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

In [11]:
train_df[target] = train_df[target].map({'No': 0, 'Yes': 1})

## Feature Engineering:

In [12]:
combined_df = pd.concat([train_df,test_df], axis=0, ignore_index=True)

### Digit Decomposition:
It breaks numeric values into individual digits and uses each digit as a separate feature.
This technique is sometimes used in tree models.

In [13]:
import numpy as np

digit_cols = []

for col in num_cols:
    max_val = combined_df[col].abs().max()

    #conditional 1:
    if max_val == 0:
        digit_count = 1
    else:
      digit_count = int(np.log10(max_val)) + 1  

    for i in range(digit_count):
        new_col = f"{col}_digit_pos_{i+1}"

        combined_df[new_col] = (combined_df[col] // (10**i)) % 10
        combined_df[new_col] = combined_df[new_col].astype(int)

        digit_cols.append(new_col)

    #conditional 2:
    if combined_df[col].dtype == 'float64':
        max_dec_count = 2

        if (combined_df[col] % 1 != 0).any():
            for i in range(1, max_dec_count + 1):
                new_col = f"{col}_digit_dec_{i}"
                
                combined_df[new_col] = (combined_df[col] * (10**i)).round().astype(int) % 10
                
                digit_cols.append(new_col)

print(f"{len(digit_cols)} Digit Features Created!!")

13 Digit Features Created!!


In [14]:
# for col in cat_cols:
#     combined_df[col] = combined_df[col].astype(str).astype('category')

In [15]:
train_df = combined_df.iloc[: len(train_df)].reset_index(drop=True)
test_df  = combined_df.iloc[len(train_df) :].reset_index(drop=True)

In [16]:
from sklearn.preprocessing import LabelEncoder

cat_cols = train_df.select_dtypes(include='object').columns.to_list()
encoder = {} #just in case we have to reuse these encoders

for col in cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = le.transform(test_df[col])
    encoder[col] = le

### Binning Transformer:

In [17]:
from sklearn.base import BaseEstimator, TransformerMixin

class Binning(BaseEstimator, TransformerMixin):
    def __init__(self, col_to_bin, num_bins, new_col_name ,labels=None):
        self.col_to_bin = col_to_bin
        self.num_bins = num_bins
        self.labels = labels
        self.new_col_name = new_col_name

    def fit(self, X, y=None):
        X = X.copy()
        _, self.bin_edges = pd.cut(X[self.col_to_bin], bins=self.num_bins, labels=False, retbins=True) #the last attribute ensures that the same bins are used to bin teh test data during transform
        return self

    def transform(self,X):
        X = X.copy() 
        X[self.new_col_name] = pd.cut(X[self.col_to_bin], bins=self.bin_edges, labels=False)
        return X

### GroupMean Encoder:
Applying group-mean encoding on Monthly and Total Charges columns, by grouping them using 'tenure bins' created using binning transformer

In [18]:
class GroupMeanEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, groupby_col, agg_col, new_col_name):
        self.groupby_col = groupby_col
        self.agg_col = agg_col
        self.new_col_name = new_col_name

    def fit(self,X,y=None):
        self.means = X.groupby(self.groupby_col,observed=True)[self.agg_col].mean()
        return self

    def transform(self,X):
        X = X.copy()
        X[self.new_col_name] = X[self.groupby_col].map(self.means)
        return X

### Frequency Encoding:
Frequency Encoding is a way to convert a categorical feature into numbers by replacing each category with how often it appears in the dataset.

In [19]:
class FreqEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols, normalize=True):
        self.cat_cols = cat_cols
        self.normalize = normalize
        self.freq_maps = {}

    def fit(self, X, y=None):
        for col in self.cat_cols:
            self.freq_maps[col] = X[col].value_counts(normalize=self.normalize)
        return self

    def transform(self, X):
        X = X.copy()

        for col in self.cat_cols:
            X[col + '_freq'] = X[col].map(self.freq_maps[col])
            X[col + '_freq'] = X[col + '_freq'].fillna(0) 
        return X

### Preprocessing Pipeline:

In [20]:
from sklearn.pipeline import Pipeline

preprocessor = Pipeline([
    ('Binning', Binning(col_to_bin='tenure', num_bins=3, new_col_name='tenure_bins')),
    ('GroupMeanEncoder_BP', GroupMeanEncoder(groupby_col='tenure_bins', agg_col='MonthlyCharges', new_col_name='X1')),
    ('GroupMeanEncoder_Cholesterol', GroupMeanEncoder(groupby_col='tenure_bins', agg_col='TotalCharges', new_col_name='X2')),
    ('FreqEncoding', FreqEncoder(cat_cols=cat_cols)),
])

## Model Training:

In [21]:
features = cat_cols + num_cols + digit_cols

In [22]:
best_xgb_params = {'n_estimators': 3488, 
                   'learning_rate': 0.04215348921376158,
                   'max_depth':3, 
                   'min_child_weight': 3, 
                   'gamma': 0.373740919749451, 
                   'subsample': 0.8908198617400997, 
                   'colsample_bytree': 0.7125429511845258,
                   'reg_alpha': 5.2763994068891605e-08, 
                   'reg_lambda': 1.0023283765023319e-07}

In [23]:
from xgboost import XGBClassifier

best_model = XGBClassifier(
    **best_xgb_params,
    random_state= 42,
    eval_metric= "logloss",
    tree_method= "hist",   
    verbosity= 0,
    device='cuda',
    enable_categorical=True
)

final_pipeline = Pipeline([
        ('prep', preprocessor),
        ('model', best_model)
    ])

final_pipeline.fit(train_df[features],train_df[target])

Pipeline(steps=[('prep',
                 Pipeline(steps=[('Binning',
                                  Binning(col_to_bin='tenure',
                                          new_col_name='tenure_bins',
                                          num_bins=3)),
                                 ('GroupMeanEncoder_BP',
                                  GroupMeanEncoder(agg_col='MonthlyCharges',
                                                   groupby_col='tenure_bins',
                                                   new_col_name='X1')),
                                 ('GroupMeanEncoder_Cholesterol',
                                  GroupMeanEncoder(agg_col='TotalCharges',
                                                   groupby_col='tenure_bins',
                                                   new_col_name='X2')),
                                 ('...
                               gamma=0.373740919749451, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.04215348921376158, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=3,
                               max_leaves=None, min_child_weight=3, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=3488, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [24]:
y_pred = final_pipeline.predict_proba(test_df[features])[:,1]

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [25]:
submission = pd.DataFrame({
    'id': test_df['id'],
    target: y_pred
})

submission.to_csv('submission.csv', index=False)